# Sprint 3 — Coding Attention Mechanisms
Demonstração incremental que reutiliza os componentes reais da Sprint 2. As projeções ainda não foram treinadas; os pesos abaixo demonstram somente o funcionamento matemático.

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.tokenization import create_simple_tokenizer, tokenize_text
from src.data import create_dataloader_v1
from src.embeddings import TokenAndPositionEmbedding
from src.attention import SelfAttention, CausalAttention, MultiHeadAttention, scaled_dot_product_attention

torch.manual_seed(123)

## 1. Carregar o texto

In [ ]:
text = (ROOT / 'data' / 'texto_teste.txt').read_text(encoding='utf-8')
print('Caracteres no corpus:', len(text))
print(text[:140], '...')

## 2. Tokenizar

In [ ]:
tokenizer = create_simple_tokenizer(text)
example = 'Um modelo de linguagem aprende padrões.'
print('Tokens:', tokenize_text(example))
print('Token IDs:', tokenizer.encode(example))
print('Vocabulário:', tokenizer.vocab_size)

## 3. Criar o DataLoader

In [ ]:
context_length = 8
loader = create_dataloader_v1(
    text, tokenizer, batch_size=2, max_length=context_length,
    stride=4, shuffle=False, drop_last=False
)
input_ids, target_ids = next(iter(loader))
print('input_ids:', input_ids.shape, '= [B, T]')
print('target_ids:', target_ids.shape)

## 4. Gerar Token Embeddings + Positional Embeddings

In [ ]:
embedding_dim = 16
embedding_layer = TokenAndPositionEmbedding(
    tokenizer.vocab_size, embedding_dim, context_length
)
embeddings = embedding_layer(input_ids)
print('embeddings:', embeddings.shape, '= [B, T, D]')

## 5. Self-Attention
Cada posição consulta todas as posições. Esta primeira versão não aplica máscara causal.

In [ ]:
self_attention = SelfAttention(d_in=embedding_dim, d_out=embedding_dim)
self_context, self_weights = self_attention(embeddings, return_attention_weights=True)
print('context:', self_context.shape, '= [B, T, D]')
print('attention weights:', self_weights.shape, '= [B, T, T]')
print('Somas das linhas:', self_weights[0].sum(dim=-1))

## 6. Mostrar Q, K e V

In [ ]:
queries, keys, values = self_attention.project_qkv(embeddings)
print('Q:', queries.shape)
print('K:', keys.shape)
print('V:', values.shape)

## 7. Scaled Dot-Product Attention
A função calcula explicitamente `softmax(QKᵀ / sqrt(d_k))V`. A escala reduz a saturação do softmax em dimensões maiores.

In [ ]:
scaled_context, scaled_weights = scaled_dot_product_attention(queries, keys, values)
scores = queries @ keys.transpose(-2, -1)
print('scores QK^T:', scores.shape, '= [B, T, T]')
print('scaled context:', scaled_context.shape)
print('Resultado igual ao forward:', torch.allclose(scaled_context, self_context))

## 8. Mostrar a máscara causal

In [ ]:
causal_attention = CausalAttention(
    embedding_dim, embedding_dim, context_length, dropout=0.0
)
print(causal_attention.causal_mask.to(torch.int32))
print('1 = permitido; 0 = posição futura bloqueada')

## 9. Causal Attention

In [ ]:
causal_context, causal_weights = causal_attention(embeddings, return_attention_weights=True)
future = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
print('causal context:', causal_context.shape)
print('causal weights:', causal_weights.shape)
print('Maior peso futuro:', causal_weights[:, future].max().item())

## 10. Multi-Head Attention

In [ ]:
multi_head = MultiHeadAttention(
    d_in=embedding_dim, d_out=embedding_dim, context_length=context_length,
    dropout=0.0, num_heads=4
)
multi_q, multi_k, multi_v = multi_head.project_qkv(embeddings)
multi_output, multi_weights = multi_head(embeddings, return_attention_weights=True)
print('Q/K/V separados:', multi_q.shape, '= [B, H, T, head_dim]')
print('Pesos:', multi_weights.shape, '= [B, H, T, T]')
print('Output:', multi_output.shape, '= [B, T, D]')
print('head_dim:', multi_head.head_dim)

## 11. Visualizar matrizes
O eixo X contém as keys observadas; o eixo Y contém as queries. A região futura permanece zerada.

In [ ]:
labels = [tokenizer.int_to_str[int(token_id)] for token_id in input_ids[0]]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for head_index, axis in enumerate(axes.flat):
    image = axis.imshow(multi_weights[0, head_index].detach(), cmap='viridis', vmin=0)
    axis.set_title(f'Head {head_index + 1}')
    axis.set_xlabel('Keys')
    axis.set_ylabel('Queries')
    axis.set_xticks(range(context_length), labels, rotation=45, ha='right')
    axis.set_yticks(range(context_length), labels)
    fig.colorbar(image, ax=axis)
fig.suptitle('Multi-Head Causal Attention — parâmetros não treinados')
fig.tight_layout()
plt.show()

## 12. Resumo das dimensões

- Token IDs: `[B,T]`
- Embeddings: `[B,T,D]`
- Q/K/V de uma head: `[B,T,D]`
- Scores de uma head: `[B,T,T]`
- Q/K/V separados: `[B,H,T,head_dim]`
- Pesos Multi-Head: `[B,H,T,T]`
- Saída concatenada e projetada: `[B,T,D]`

A última saída preserva a interface esperada pelos Transformer Blocks da Sprint 4.